In [1]:
import torch
import torch.nn as nn
import numpy as np

def im2col_multi(X, kernel_shape, stride=1, padding=(0, 0)):
    B = X.shape[0]
    kH, kW = kernel_shape

    if isinstance(padding, tuple):
        pad_H, pad_W = padding
    else:
        pad_H = pad_W = padding

    X_padded = np.pad(X, ( (0, 0), (0, 0),(pad_H, pad_H), (pad_W, pad_W) ), mode='constant')

    H_p, W_p = X_padded.shape[2:]

    out_H = (H_p- kH) // stride + 1
    out_W = (W_p- kW) // stride + 1

    cols = []

    for b in range(B):
        for i in range(0, out_H*stride, stride):
            for j in range(0, out_W * stride, stride):
                patch = X_padded[b, :, i:i+kH, j:j+kW].ravel()
                cols.append(patch)
    
    return np.array(cols), out_H, out_W

def col2im_multi(cols, output_shape, kernel_shape, stride=1, padding=0):
    B, C, H, W = output_shape
    kH, kW = kernel_shape
    H_p, W_p = H+2*padding, W+2*padding
    X_padded = np.zeros((B, C, H_p, W_p))

    out_H = (H_p - kH)//stride + 1
    out_W = (W_p - kW)//stride + 1

    idx = 0
    for b in range(B):
        for i in range(0, out_H*stride, stride):
            for j in range(0, out_W*stride, stride):
                patch = cols[idx].reshape(C, kH, kW)
                X_padded[b, :, i:i+kH, j:j+kW] += patch
                idx += 1

    if padding>0:
        X_padded = X_padded[:, :, padding:-padding, padding:-padding]

    return X_padded

def conv2d_im2col_multi(X, W, stride=1, padding=0):
    
    C_out, C_in, kH, kW = W.shape
    B = X.shape[0]
    X_col, out_H, out_W = im2col_multi(X, (kH, kW), stride, padding)

    W_col = W.reshape(C_out, -1)
    Y_col = X_col @ W_col.T

    Y = Y_col.T.reshape(B, C_out, out_H, out_W)
    return Y

def conv_transpose2d_img2col_multi(Y, W, stride=1, padding=0, output_shape=None):
    C_out, C_in, kH, kW = W.shape
    B = Y.shape[0]
    Y_col = Y.reshape(C_out, -1)
    W_col = W.reshape(C_out, -1)
    X_col = W_col.T @ Y_col

    if output_shape is None:
        H_out = (Y.shape[2]-1) * stride - 2*padding + kH
        W_out = (Y.shape[3]-1) * stride - 2*padding + kW
        output_shape = (B, C_in, H_out, W_out)

    X = col2im_multi(X_col.T, output_shape=output_shape, kernel_shape=(kH, kW), stride=stride, padding=padding)

    return X

In [2]:
torch.manual_seed(1)

B= 10
C_out, C_in = 6, 3
input_size=15
kernel_size=3

stride=1
padding=0

x = torch.randn(B, C_in, input_size, input_size)
kernel = torch.ones(C_out, C_in, kernel_size, kernel_size)

print(f'x: {x.shape}')
print(f'Kernel: {kernel.shape}')

x: torch.Size([10, 3, 15, 15])
Kernel: torch.Size([6, 3, 3, 3])


In [3]:
conv = nn.Conv2d(C_in, C_out, kernel_size=kernel_size, stride=stride, padding=padding)
convt = nn.ConvTranspose2d(6, 3, kernel_size=kernel_size, stride=stride, padding=padding)

with torch.no_grad():
    conv.weight.copy_(kernel)
    conv.bias.zero_()
    convt.weight.copy_(kernel)
    convt.bias.zero_()

output_conv = conv(x)
print(f'Conv out: {output_conv.shape}')

output_convt = convt(output_conv)
print(f'ConvT out: {output_convt.shape}')

Conv out: torch.Size([10, 6, 13, 13])
ConvT out: torch.Size([10, 3, 15, 15])


In [4]:
x = np.array(x)
kernel = np.array(kernel)

result = conv2d_im2col_multi(x, kernel, stride=stride, padding=padding)
print(f'result: {result.shape}')

original = conv_transpose2d_img2col_multi(result, kernel, stride=stride, padding=padding, output_shape=None)
print(f'original: {original.shape}')
#print(original)

result: (10, 6, 13, 13)
original: (10, 3, 15, 15)


C:\Users\Korisnik\AppData\Local\Temp\ipykernel_11748\1603341396.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  x = np.array(x)
C:\Users\Korisnik\AppData\Local\Temp\ipykernel_11748\1603341396.py:2: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  kernel = np.array(kernel)


In [5]:
print(output_conv[0, 0, 0])
print(result[0, 0, 0])

tensor([ -0.7755,   0.7778,   5.9921,   6.4423,   6.5881,  -0.2533,  -6.0095,
         -6.7197, -10.1001,  -4.2571, -13.5088, -10.1428,  -5.7500],
       grad_fn=<SelectBackward0>)
[ -0.77549046   0.7778102    5.992073     6.4423437    6.5881467
  -0.25333259  -6.0095124   -6.719651   -10.10014     -4.257121
 -13.508778   -10.142757    -5.749958  ]


In [6]:
print(output_convt[0, 0,0 ])
print(original[0, 0, 0])

tensor([-4.6529e+00,  1.3919e-02,  3.5966e+01,  7.9273e+01,  1.1414e+02,
         7.6663e+01,  1.9518e+00, -7.7895e+01, -1.3698e+02, -1.2646e+02,
        -1.6720e+02, -1.6745e+02, -1.7641e+02, -9.5356e+01, -3.4500e+01],
       grad_fn=<SelectBackward0>)
[-4.65294266e+00  1.39183998e-02  3.59663568e+01  7.92733636e+01
  1.14135384e+02  7.66629497e+01  1.95181000e+00 -7.78949765e+01
 -1.36975819e+02 -1.26461472e+02 -1.67196224e+02 -1.67451931e+02
 -1.76408951e+02 -9.53562927e+01 -3.44997482e+01]
